# Audio Editor
## This notebook outlines the techniques used to edit the audio signal (removing long silences)

### Import the libraries

In [1]:
from pydub import AudioSegment
import numpy as np

### Read MP3 file using PyDub

In [2]:
audiofile = AudioSegment.from_wav("sample.wav")

In [3]:
audiofile

### Convert the samples into Numpy array

In [4]:
data_mp3 = np.array(audiofile.get_array_of_samples())

In [5]:
data_mp3.shape

(524188,)

### Get the frame rate

In [6]:
fs_mp3 = audiofile.frame_rate

In [7]:
fs_mp3

44100

### Find the signal duration

In [8]:
print(f'Signal Duration = {data_mp3.shape[0] / (2 * fs_mp3)} seconds')

Signal Duration = 5.9431746031746036 seconds


### Read a wav file
- Use scipy's wavfile

In [9]:
from scipy.io import wavfile
fs_wav, data_wav = wavfile.read("sample.wav")

/var/folders/g9/p5jy_4tn7kv289h_f8z65tg40000gn/T/ipykernel_33592/2074904232.py:2: WavFileWarning: Chunk (non-data) not understood, skipping it.
  fs_wav, data_wav = wavfile.read("sample.wav")


### Get the frame rate

In [10]:
fs_wav

44100

In [11]:
data_wav.shape

(262094, 2)

### Get the duration of the signal

In [12]:
print(f'Signal Duration = {data_wav.shape[0] / fs_wav} seconds')

Signal Duration = 5.9431746031746036 seconds


### Normalize the signal

In [13]:
data_wav_norm = data_wav / (2**15)

### Length of the signal

In [14]:
signal_len = len(data_wav_norm)
signal_len

262094

### Fix segment size in seconds

In [15]:
segment_size_t = 1 

### Segment size in samples

In [16]:
segment_size = segment_size_t * fs_wav

### Split audio signal into 1 second segments

In [17]:
segments = np.array([data_wav_norm[x:x + segment_size] for x in
                     np.arange(0, signal_len, segment_size)], dtype=object)

### How many segments ?

In [18]:
segments.shape[0]

6

### Save each segment in a seperate filename

In [19]:
import os
os.makedirs("segments", exist_ok=True)

for iS, s in enumerate(segments):
    wavfile.write("segments/sample_segment_{0:d}_{1:d}.wav".format(
        segment_size_t * iS, segment_size_t * (iS + 1)), fs_wav, (s))

### Remove pauses using an energy threshold = 50% of the median energy

In [20]:
energies = [(s**2).sum() / len(s) for s in segments]

### Fix 50% as threshold

In [21]:
thres = 0.5 * np.median(energies)

### Collect the indexes of segments which is above this 50% threshold

In [22]:
index_of_segments_to_keep = (np.where(energies > thres)[0])

### Get segments that have energies higher than the threshold

In [23]:
segments2 = segments[index_of_segments_to_keep]

### Concatenate segments to signal

In [24]:
new_signal = np.concatenate(segments2)

### Write the file

In [25]:
wavfile.write("sample_processed.wav", fs_wav, new_signal)

## ASSIGNMENT: Take any mp3 file containing speeches of someone of your choice with pauses inbetween the speech and apply the above techniques to remove pauses. Become an audio editor!
- Submit the mp3 file of original speech, notebook file, mp3 file of edited speech